# Week 4 scratchpad: tool design, context management, MCP

Concept first, then a small snippet that shows it. Read top to bottom.

| Part | Topic                         | Runs offline? |
|------|-------------------------------|---------------|
| 0    | Setup                         | yes           |
| 1    | Tool design                   | yes           |
| 2    | Context management            | yes           |
| 3    | MCP concepts + raw wire demo  | yes           |
| 4    | MCP vs local function         | (text only)   |
| 5    | HR leave example, end to end  | yes, except cells marked **[$]** (call Claude, cost money) |
| 6    | Cheat sheet                   | (text only)   |

Code for part 5 lives in `../hr-leave-mcp/`.

## 0. Setup

In [1]:
# pip install "mcp>=2.3" anthropic fastapi uvicorn httpx python-dotenv
import json, sys, subprocess, time, socket
from pathlib import Path

HR_DIR = Path("../hr-leave-mcp").resolve()    # the part-5 project
print("python:", sys.version.split()[0], "| hr-leave-mcp found:", HR_DIR.exists())

python: 3.13.18 | hr-leave-mcp found: True


## 1. Tool design

**Why:** the model only sees a tool's name, description, inputs, and what it returns. Bad tools mean wasted steps, wrong calls, and a bloated context.

**Four rules**
1. **Fewer, higher-level tools.** One `search_logs(service, pattern)` beats `list_files` + `read_file` + `grep`.
2. **Actionable errors.** Say what went wrong *and* what to do next.
3. **Compact output.** Return only what helps the decision. Every byte is re-sent on every later turn.
4. **Pagination.** Return the first N and say how to get more.

**Cautions:** don't merge into a vague `do_everything` tool. Truncation can hide the key line, so always say it was cut. The description is a prompt: fix it first when the model misuses a tool.

### 1a. Bad vs good error message

In [2]:
SERVICES = ["checkout", "payments", "search"]

def search_logs_bad(service):
    if service not in SERVICES:
        return "Error: 404"                       # model has to guess what went wrong

def search_logs_good(service):
    if service not in SERVICES:
        return f"Unknown service '{service}'. Valid: {', '.join(SERVICES)}."   # model can fix the next call

print("bad :", search_logs_bad("checkout-svc"))
print("good:", search_logs_good("checkout-svc"))

bad : Error: 404
good: Unknown service 'checkout-svc'. Valid: checkout, payments, search.


### 1b. Compact output + pagination

In [3]:
LOGS = [f"09:{i//60:02d}:{i%60:02d} ERROR db pool exhausted req={i} trace=ab{i:06x}..." for i in range(4000)]

def search_logs(pattern: str, page: int = 1, page_size: int = 50) -> str:
    hits = [l for l in LOGS if pattern.lower() in l.lower()]
    start = (page - 1) * page_size
    shown = [l.split(" trace=")[0] for l in hits[start:start + page_size]]   # compact: drop trace ids
    footer = f"[{len(hits)} matches, showing {start+1}-{start+len(shown)}. Narrow 'pattern' or pass page={page+1}.]"
    return "\n".join(shown[:3]) + "\n...\n" + footer      # print only 3 lines here to keep output short

print(search_logs("pool"))

09:00:00 ERROR db pool exhausted req=0
09:00:01 ERROR db pool exhausted req=1
09:00:02 ERROR db pool exhausted req=2
...
[4000 matches, showing 1-50. Narrow 'pattern' or pass page=2.]


## 2. Context management on long runs

**Why:** the API is stateless, so every turn re-sends the whole history. By step 12, most of it is old tool output the model already used. It costs money, slows calls, and buries the important bits.

**Three techniques, cheapest first**
1. **Clear old tool results.** Swap a used result's body for one line. Keep the `tool_use`/`tool_result` pair so the structure stays valid. Trigger past a token threshold. (Claude's API can do this server-side with *context editing*.)
2. **Compaction.** Summarise the history into a short brief and continue from it.
3. **Notes scratchpad.** A `write_note` tool. Notes survive clearing and compaction.

**Cautions:** a summary can drop the one line that mattered, so keep pointers (file, time range) for re-fetching. Editing early messages breaks prompt caching from that point, so clear in batches, not every turn.

### 2a. Clear old tool results (offline, on a fake history)

In [4]:
def approx_tokens(messages) -> int:
    return len(json.dumps(messages)) // 4          # rough: ~4 chars per token

def clear_old_tool_results(messages, keep_last=2, threshold=2_000):
    """Past `threshold` tokens, replace all but the last `keep_last` tool results with a one-liner."""
    if approx_tokens(messages) < threshold:
        return messages
    result_idx = [i for i, m in enumerate(messages)
                  if m["role"] == "user" and isinstance(m["content"], list)
                  and m["content"] and m["content"][0].get("type") == "tool_result"]
    for i in result_idx[:-keep_last]:
        for block in messages[i]["content"]:
            body = block["content"]
            if not body.startswith("[cleared]"):
                block["content"] = f"[cleared] {len(body)} chars. First line: {body.splitlines()[0][:80]}"
    return messages

# fake 5-step investigation, each tool result ~3,000 chars
history = [{"role": "user", "content": "Checkout 503s since 09:07. Why?"}]
for step in range(5):
    history.append({"role": "assistant", "content": [{"type": "tool_use", "id": f"t{step}", "name": "search_logs", "input": {}}]})
    history.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{step}",
                    "content": f"09:0{step} ERROR pool exhausted step={step}\n" + "noise line\n" * 270}]})

before = approx_tokens(history)
clear_old_tool_results(history)
print(f"tokens before: ~{before:,}   after: ~{approx_tokens(history):,}")
print("step 0 result now:", history[2]["content"][0]["content"])

tokens before: ~4,357   after: ~1,951
step 0 result now: [cleared] 3004 chars. First line: 09:00 ERROR pool exhausted step=0


### 2b. Notes scratchpad (a local tool, not MCP)

In [5]:
NOTES: list[str] = []

def write_note(text: str) -> str:
    """Record a finding you will need later. Notes survive context clearing."""
    NOTES.append(text)
    return f"Saved note #{len(NOTES)}."

write_note("deploy d-41 at 09:05 changed checkout db pool 50 -> 5")
write_note("first pool exhausted error at 09:07")
# The host re-injects notes each turn, e.g. in the latest user message:
print("<notes>\n" + "\n".join(f"- {n}" for n in NOTES) + "\n</notes>")

<notes>
- deploy d-41 at 09:05 changed checkout db pool 50 -> 5
- first pool exhausted error at 09:07
</notes>


## 3. MCP concepts

**The problem:** without a standard, every AI app (Slack bot, Claude Desktop, another team's agent) writes its own copy of the same tools. **MCP** (Model Context Protocol) is a standard way for any app to discover and call tools that live in a separate program. Write the tools once, use them from any host.

**Roles**
- **Host:** the app with the LLM loop (your agent, Claude Code, Claude Desktop).
- **MCP client:** the connector inside the host, one per server. It comes from the SDK (`ClientSession`).
- **MCP server:** the program that owns the tools *and runs them*. It never sees the LLM.

**The 4 layers of the protocol**

| Layer      | What it is                                                         |
|------------|--------------------------------------------------------------------|
| Roles      | host, client, server                                               |
| Transport  | **stdio** (local child process) or **Streamable HTTP** (shared URL)|
| Format     | JSON-RPC 2.0: request with `id`, response with same `id`, notification without `id` |
| Lifecycle  | `initialize` → `notifications/initialized` → `tools/list` → `tools/call` … → close |

**What a server can offer, by who decides**

| Primitive | Who decides | Example                      |
|-----------|-------------|------------------------------|
| Tools     | the model   | `search_logs(service)`       |
| Resources | the host    | `runbook://checkout`         |
| Prompts   | the user    | "/investigate INC-001"       |

**Spec vs SDK:** the spec (modelcontextprotocol.io) is the standard. The `mcp` package is one implementation, like HTTP vs `requests`. A Python server works with a TypeScript client.

**Who talks to whom (one question)**
```mermaid
sequenceDiagram
    participant U as User
    participant H as Host (agent)
    participant C as Claude
    participant S as MCP server
    H->>S: initialize + tools/list (once, at connect)
    U->>H: question
    H->>C: question + tools
    C->>H: tool_use search_logs(...)
    H->>S: tools/call search_logs(...)
    S->>H: result
    H->>C: tool_result
    C->>H: final answer
    H->>U: answer
```
Claude never talks to the server. The server never sees Claude.

### 3a. See the raw wire: talk JSON-RPC to a server by hand

No MCP client library. We start a tiny server as a child process (stdio) and write JSON lines to its stdin. This is exactly what `ClientSession` does for you.

In [6]:
Path("tiny_server.py").write_text('''
from mcp.server.mcpserver import MCPServer
mcp = MCPServer("tiny")

@mcp.tool()
def add(a: int, b: int) -> str:
    """Add two integers."""
    return str(a + b)

mcp.run()          # stdio transport
''')

proc = subprocess.Popen([sys.executable, "tiny_server.py"], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

def send(msg):  proc.stdin.write(json.dumps(msg) + "\n"); proc.stdin.flush()
def recv():     return json.loads(proc.stdout.readline())

# (a) handshake
send({"jsonrpc": "2.0", "id": 1, "method": "initialize",
      "params": {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "by-hand", "version": "0"}}})
print("initialize ->", json.dumps(recv()["result"])[:160], "...")
send({"jsonrpc": "2.0", "method": "notifications/initialized"})          # notification: no id, no reply

# (b) discovery
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
tool = recv()["result"]["tools"][0]
print("tools/list ->", tool["name"], "|", tool["description"], "|", tool["inputSchema"]["properties"])

# (c) use
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "add", "arguments": {"a": 2, "b": 3}}})
print("tools/call ->", recv()["result"]["content"])

proc.kill()

initialize -> {"capabilities": {"prompts": {"listChanged": false}, "resources": {"listChanged": false, "subscribe": false}, "tools": {"listChanged": false}}, "protocolVersion ...
tools/list -> add | Add two integers. | {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
tools/call -> [{'text': '5', 'type': 'text'}]


### 3b. Same thing with the SDK client

`ClientSession` builds those JSON messages for you. Note `stdio_client` *spawns* the server: with stdio, the host starts the server.

In [7]:
from mcp import ClientSession, StdioServerParameters, stdio_client

async def demo_sdk_client():
    params = StdioServerParameters(command=sys.executable, args=["tiny_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()                         # initialize + initialized
            tools = await session.list_tools()                 # tools/list
            out = await session.call_tool("add", {"a": 2, "b": 3})   # tools/call
            print([t.name for t in tools.tools], "->", out.content[0].text)

await demo_sdk_client()

['add'] -> 5


### 3c. MCP tool → Claude tool

The host converts what `tools/list` returned into Claude's `tools` format. Same three fields.

In [8]:
def to_claude_tool(t) -> dict:
    return {"name": t.name, "description": t.description or "", "input_schema": t.input_schema}

mcp_tool_from_list = {"name": "add", "description": "Add two integers.",
                      "inputSchema": {"type": "object", "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}}, "required": ["a", "b"]}}
from types import SimpleNamespace
print(to_claude_tool(SimpleNamespace(name=mcp_tool_from_list["name"], description=mcp_tool_from_list["description"],
                                     input_schema=mcp_tool_from_list["inputSchema"])))
# (The anthropic SDK also has a beta helper for this: anthropic.lib.tools.mcp.async_mcp_tool)

{'name': 'add', 'description': 'Add two integers.', 'input_schema': {'type': 'object', 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}}


**Cautions**
- MCP doesn't make tools better. Part 1 rules still apply.
- A third-party server runs code with your permissions, and its tool descriptions go into your prompt. Treat it as untrusted.
- With stdio, never `print()` to stdout in the server: it corrupts the protocol. Log to stderr.
- `mcp` 2.x renamed `FastMCP` to `MCPServer`. Raise `ToolError` for messages the model should see; other exceptions are hidden.

## 4. MCP vs local function

MCP adds a process, a protocol, and a failure point. Worth it only when something besides your one agent needs the tools.

| Use MCP when                                        | Use a local function when                         |
|-----------------------------------------------------|---------------------------------------------------|
| More than one host uses the tools                   | Only your agent calls it                          |
| Another team owns the system (they ship the server) | It's tightly coupled to your loop (`write_note`)  |
| Tools need isolation (own credentials, own machine) | Latency or simplicity matters more                |
| A ready-made server exists (GitHub, Postgres)       | You're still prototyping                          |

**Rule of thumb:** start local, move to MCP when a second consumer shows up. Connect only the servers a task needs: every server's tool schemas land in the prompt.

## 5. HR leave example, end to end

**Scenario:** the HR system has leave data. Three AI apps want it: a Slack bot, Claude Desktop, an onboarding agent. Without MCP, each writes its own leave tools.

**Who owns what**

| Team            | Owns                                  | Files in `hr-leave-mcp/`               |
|-----------------|---------------------------------------|----------------------------------------|
| HR team         | HR system (already exists)            | `hr_api.py`                            |
| HR team         | MCP server = tools + running them     | `mcp_server.py` + `leave_tools.py` + `auth.py` |
| App team        | Host = LLM loop + MCP client          | `host_agent.py`                        |
| (testing only)  | Plain checks, no LLM                  | `try_tools.py`, `inspect_server.py`    |

```mermaid
flowchart LR
    U[Employee] --> H[host_agent.py<br/>LLM loop + MCP client]
    H <-->|HTTPS| C[Claude]
    H <-->|MCP over HTTP<br/>+ Bearer token| S[mcp_server.py<br/>:8002/mcp]
    S --> T[leave_tools.py]
    T -->|REST| A[hr_api.py<br/>:8001]
```

Two deployables: `hr_api` and `mcp_server` (with `leave_tools` inside it).

### 5.1 Step 1: the HR API and the tool wrappers (no MCP, no LLM)

- `hr_api.py`: a normal REST service for programs. `GET /employees/{emp}/balance`, `POST /employees/{emp}/leave`. JSON + status codes.
- `leave_tools.py`: the model-facing wrappers. Validate inputs, call the API, return short text, turn HR errors into actionable messages.

Start the HR API (it keeps running in the background until part 5.6):

In [9]:
def wait_port(port, timeout=10):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(0.2)
    raise RuntimeError(f"nothing listening on {port}")

hr_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "hr_api:app", "--port", "8001"],
                           cwd=HR_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
wait_port(8001); print("HR API up on :8001")

HR API up on :8001


Call the tool functions directly, like the model would:

In [10]:
sys.path.insert(0, str(HR_DIR))
from leave_tools import get_leave_balance, request_leave, LeaveToolError

for label, fn in [
    ("balance",        lambda: get_leave_balance("E042")),
    ("bad type",       lambda: get_leave_balance("E042", "vacation")),
    ("too many days",  lambda: request_leave("E077", "casual", "2026-12-21", "2026-12-23")),
    ("bad date",       lambda: request_leave("E042", "casual", "14/12/2026", "14/12/2026")),
]:
    try:    print(f"OK    {label:14} {fn()}")
    except LeaveToolError as e: print(f"ERROR {label:14} {e}")

OK    balance        casual: 4, sick: 6, earned: 12 (as of 2026-11-11)
ERROR bad type       Unknown leave_type 'vacation'. Valid: casual, sick, earned, or leave it empty for all.
ERROR too many days  Not enough casual leave: requested 3, available 1. Call get_leave_balance to see other types, or shorten the dates.
ERROR bad date       start_date '14/12/2026' is not a valid date. Use YYYY-MM-DD, e.g. 2026-11-14.


Note: `employee_id` is a function argument here only for testing. It is **never** a tool input the model fills in (step 4).

### 5.2 Step 2: the MCP server

An MCP server answers two questions from any app: **"what tools do you have?"** (`tools/list`) and **"run this one"** (`tools/call`). The key part of `mcp_server.py`:

```python
mcp = MCPServer("hr-leave", instructions="...", token_verifier=DemoTokenVerifier(), auth=AuthSettings(...))

LeaveType = Literal["casual", "sick", "earned"]

@mcp.tool()
def get_leave_balance(leave_type: LeaveType | None = None) -> str:
    """Get the signed-in employee's remaining leave days. Omit leave_type to get all types."""
    try:
        return leave_tools.get_leave_balance(current_employee(), leave_type or "")
    except LeaveToolError as e:
        raise ToolError(str(e))

mcp.run(transport="streamable-http", port=8002)
```
- Docstring → description. Type hints → input schema (`Literal` → `enum`).
- No `anthropic` import. The server doesn't know an LLM exists.
- `transport="streamable-http"`: one shared server at a URL, because many apps and employees use it. A personal/local tool would use stdio instead.

Start it:

In [11]:
mcp_proc = subprocess.Popen([sys.executable, "mcp_server.py"], cwd=HR_DIR,
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
wait_port(8002); print("MCP server up on http://127.0.0.1:8002/mcp")

MCP server up on http://127.0.0.1:8002/mcp


### 5.3 Step 3: a plain MCP client (no LLM)

What every host does first: handshake → `tools/list` → `tools/call`. Over HTTP this time, with a login token in the header.

In [12]:
import httpx2
from mcp.client.streamable_http import streamable_http_client

MCP_URL = "http://127.0.0.1:8002/mcp"

async def call_as(token: str, name: str, args: dict):
    http = httpx2.AsyncClient(headers={"Authorization": f"Bearer {token}"}, timeout=30)
    async with http, streamable_http_client(MCP_URL, http_client=http) as (read, write, *_):
        async with ClientSession(read, write) as session:
            await session.initialize()
            out = await session.call_tool(name, args)
            return out.is_error, out.content[0].text

async def list_tools_as(token: str):
    http = httpx2.AsyncClient(headers={"Authorization": f"Bearer {token}"}, timeout=30)
    async with http, streamable_http_client(MCP_URL, http_client=http) as (read, write, *_):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            tools = (await session.list_tools()).tools
            print("server:", init.server_info.name)
            for t in tools:
                print(f"- {t.name}: {t.description.splitlines()[0]}")
                print("  inputs:", {k: v.get("enum", v.get("type", v.get("anyOf"))) for k, v in t.input_schema["properties"].items()})
            return tools

hr_tools = await list_tools_as("tok-asha")

server: hr-leave
- get_leave_balance: Get the signed-in employee's remaining leave days. Omit leave_type to get all types.
  inputs: {'leave_type': [{'enum': ['casual', 'sick', 'earned'], 'type': 'string'}, {'type': 'null'}]}
- request_leave: Submit a leave request for the signed-in employee. Dates are YYYY-MM-DD, inclusive.
  inputs: {'leave_type': ['casual', 'sick', 'earned'], 'start_date': 'string', 'end_date': 'string', 'reason': 'string'}


In [13]:
print(await call_as("tok-asha", "get_leave_balance", {}))
print(await call_as("tok-asha", "get_leave_balance", {"leave_type": "vacation"}))   # rejected by the schema, before our code

(False, 'casual: 4, sick: 6, earned: 12 (as of 2026-11-11)')
(True, "Error executing tool get_leave_balance: 1 validation error for get_leave_balanceArguments\nleave_type\n  Input should be 'casual', 'sick' or 'earned' [type=literal_error, input_value='vacation', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/literal_error")


### 5.4 Step 4: identity (who is asking?)

One shared server, many employees. The server knows who's calling from the **login token on every request**, never from the model.

1. Employee logs in once via company SSO (OAuth). The host gets a token.
2. Host sends `Authorization: Bearer <token>` on every MCP request.
3. Server verifies it (`DemoTokenVerifier` here; a signed JWT in real life) and reads the employee + scopes.
4. `current_employee()` uses that. No `employee_id` input exists for the model to abuse.

```python
def current_employee(need_scope="leave:read") -> str:
    token = get_access_token()             # set by the SDK after verify_token() passed
    if need_scope not in token.scopes:
        raise ToolError(f"This login can't do that (missing '{need_scope}')...")
    return token.subject                   # e.g. "E042"
```
Same tool call, different tokens:

In [14]:
call = ("request_leave", {"leave_type": "earned", "start_date": "2026-12-28", "end_date": "2026-12-28"})
for token in ["tok-asha", "tok-rahul", "tok-asha-read"]:
    is_err, text = await call_as(token, *call)
    print(f"{token:14} is_error={is_err!s:5} {text}")

try:
    await call_as("bogus", "get_leave_balance", {})
except BaseException as e:
    print("bogus          rejected before any tool ran:", type(e).__name__)

tok-asha       is_error=False Request LR-2291 submitted: earned, 2026-12-28 to 2026-12-28 (1 working day). Status: pending manager approval.


tok-rahul      is_error=False Request LR-2292 submitted: earned, 2026-12-28 to 2026-12-28 (1 working day). Status: pending manager approval.
tok-asha-read  is_error=True  Error executing tool request_leave: This login can't do that (missing 'leave:write'). Tell the user to request leave in the HR portal.
bogus          rejected before any tool ran: ExceptionGroup


### 5.5 Step 5: the host (LLM loop + MCP client)

`host_agent.py` is the week-3 tool loop. The only change: tools come from `tools/list`, and running a tool means `tools/call` to the server. Plus an approval gate before write tools.

```python
tools = [to_claude_tool(t) for t in (await session.list_tools()).tools]    # once, at connect

for _ in range(MAX_STEPS):
    resp = await claude.messages.create(model=MODEL, tools=tools, messages=messages, ...)
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        return text_of(resp)
    results = [await run_tool(session, b) for b in resp.content if b.type == "tool_use"]   # -> tools/call
    messages.append({"role": "user", "content": results})
```

First, offline with a **fake Claude** so you can watch the loop for free:

In [15]:
import host_agent
T = lambda i, n, a: SimpleNamespace(type="tool_use", id=i, name=n, input=a)

class FakeClaude:
    """Replays a fixed script of responses, so the loop runs without an API key."""
    def __init__(self, script): self.messages, self.script = self, iter(script)
    async def create(self, **kw): return next(self.script)

script = [
    SimpleNamespace(stop_reason="tool_use", content=[T("1", "get_leave_balance", {"leave_type": "casual"})]),
    SimpleNamespace(stop_reason="tool_use", content=[T("2", "request_leave", {"leave_type": "casual", "start_date": "2026-12-29", "end_date": "2026-12-29"})]),
    SimpleNamespace(stop_reason="end_turn", content=[SimpleNamespace(type="text", text="Done: casual leave on Dec 29 submitted, waiting for your manager.")]),
]

async def host_turn(claude, user_msg, token="tok-asha", auto_approve="y"):
    http = httpx2.AsyncClient(headers={"Authorization": f"Bearer {token}"}, timeout=30)
    async with http, streamable_http_client(MCP_URL, http_client=http) as (read, write, *_):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = [host_agent.to_claude_tool(t) for t in (await session.list_tools()).tools]
            messages = [{"role": "user", "content": user_msg}]
            reply = await host_agent.answer(claude, session, tools, messages,
                                            confirm=lambda p: (print(p + auto_approve), auto_approve)[1])
            print("bot>", reply)

await host_turn(FakeClaude(script), "Take casual leave on Dec 29")

  [approve] request_leave({'leave_type': 'casual', 'start_date': '2026-12-29', 'end_date': '2026-12-29'})? [y/N] y
bot> Done: casual leave on Dec 29 submitted, waiting for your manager.


  [mcp] get_leave_balance({'leave_type': 'casual'}) -> casual: 4 (as of 2026-11-11)
  [mcp] request_leave({'leave_type': 'casual', 'start_date': '2026-12-29', 'end_date': '2026-12-29'}) -> Request LR-2293 submitted: casual, 2026-12-29 to 2026-12-29 (1 working day). Status: pending manager approval.


**[$] Real Claude** (Haiku 5.5, a fraction of a cent). Needs `ANTHROPIC_API_KEY` in your env or `.env`. Prompts you to approve `request_leave` in the cell's input box.

In [16]:
# [$] costs money
# from dotenv import load_dotenv; load_dotenv()
# from anthropic import AsyncAnthropic
# await host_turn(AsyncAnthropic(), "How much casual leave do I have? If at least 1, take next Friday off.", auto_approve="y")

### 5.6 Same server, another host: Claude Code

The point of MCP: no new code for a new host. With both servers running in terminals:

```bash
claude mcp add --transport http hr-leave http://127.0.0.1:8002/mcp --header "Authorization: Bearer tok-asha"
```
Then ask Claude Code "how much leave do I have?". It runs the same `tools/list` → `tools/call` flow.

Stop the background servers:

In [17]:
for p in (mcp_proc, hr_proc):
    p.terminate()
print("stopped")

stopped


## 6. Cheat sheet

```
SERVER (tool owner, e.g. HR team)     HOST (app team, e.g. Slack bot)        CLAUDE
---------------------------------     -------------------------------        ------
@mcp.tool() def do_x(...)             session.initialize()                   picks tool + inputs
  - runs the code                     tools = session.list_tools()  (once)   never sees server
  - checks the token                  loop: claude(messages, tools)          never sees token
mcp.run(stdio | streamable-http)            tool_use -> session.call_tool()
no `anthropic` import                       result  -> tool_result
```

- **MCP** = standard connection between a host and a tool server: handshake, `tools/list`, `tools/call`.
- **Spec vs SDK:** spec at modelcontextprotocol.io; `mcp` package implements it (client = `ClientSession`, server = `MCPServer`).
- **Transport:** stdio = host spawns server locally. Streamable HTTP = one shared server at a URL the owner publishes.
- **Identity:** OAuth token from the host, verified by the server. Never a model input.
- **`tools/list`** runs once at connect (again only on a list-changed notification or reconnect).
- **The LLM decides, the host carries messages, the server acts.**
- **Tool design and context rules** (parts 1-2) still apply. MCP only changes where tools live.